# 08 · Numba CUDA ①: Copy & 메모리 접근(coalescing)

> **CuPy 2일 집중 코스 — Day 2 / 단원 6 (Numba CUDA 커널 작성, GTC 06 기반)**

이제 **Numba CUDA**(`@cuda.jit`)로 커널을 **직접** 작성합니다. 가장 단순한 *복사 커널*로
07에서 배운 **스레드 인덱싱·coalescing·occupancy** 개념을 손으로 구현하고 성능 차이를 측정합니다.

## 이 노트북에서 구현하는 개념 (07 참조)
- **2. 스레드 인덱싱**: `cuda.grid`, `blockDim/blockIdx/threadIdx`
- **5. coalescing**: blocked(흩어진) vs coalesced(연속) 접근 → 성능 비교
- **7. occupancy**: `threads_per_block`·`items_per_thread` 파라미터 스윕

## 학습 목표
- `@cuda.jit`로 커널을 정의하고 `kernel[blocks, threads](...)` 로 런치한다.
- 같은 복사 작업을 **메모리 접근 패턴**만 바꿔 가속한다.
- (선택) Nsight Compute로 메모리 처리량을 확인한다.

## 목차
1. [Numba CUDA 기초](#1)
2. [Blocked Copy (기준)](#2)
3. [Coalesced Copy (최적화)](#3)
4. [성능 비교](#4)
5. [파라미터 스윕(occupancy)](#5)
6. [(선택) Nsight Compute 프로파일](#6)
7. [체크포인트](#7)

> 필요 패키지: `numba`(CUDA 지원). `ncu`(Nsight Compute)는 6절에서만, 없으면 건너뜀.

In [ ]:
import numpy as np, cupy as cp, math
from numba import cuda
from course_utils import print_env, bench, gpu_ms, print_bench, compare
print_env()
print('Numba CUDA 사용 가능:', cuda.is_available())

<a id="1"></a>
## 1. Numba CUDA 기초

`@cuda.jit`로 장식한 Python 함수가 GPU 커널이 됩니다. 각 스레드는 `cuda.grid(1)`로 전역 인덱스를 얻습니다
(= `blockIdx.x*blockDim.x + threadIdx.x`, 07의 2절). 런치는 `kernel[그리드, 블록](인자)`.

In [ ]:
@cuda.jit
def scale_kernel(x, y, a):
    i = cuda.grid(1)              # 전역 인덱스 (2절)
    if i < x.size:               # 경계 검사
        y[i] = a * x[i]

n = 1 << 20
x = cp.arange(n, dtype=cp.float32); y = cp.empty_like(x)
threads = 256; blocks = (n + threads - 1) // threads
scale_kernel[blocks, threads](x, y, np.float32(2.0))
cp.cuda.Device().synchronize()
cp.testing.assert_allclose(cp.asnumpy(y), 2.0*cp.asnumpy(x)); print('OK')

<a id="2"></a>
## 2. Blocked Copy (기준)

각 스레드가 **연속된 `ipt`개**를 복사합니다. 한 스레드 입장에선 연속이지만, **같은 시점에 warp의 인접 스레드들은
멀리 떨어진 주소**를 읽습니다 → **흩어진(non-coalesced) 접근** (07의 5절). 메모리 효율이 낮습니다.

In [ ]:
@cuda.jit
def copy_blocked(src, dst, ipt):
    base = cuda.grid(1) * ipt
    for k in range(ipt):
        if base + k < src.size:
            dst[base + k] = src[base + k]

N = 1 << 24
src = cp.arange(N, dtype=cp.float32); dst = cp.empty_like(src)
threads = 256; ipt = 8
blocks_b = (N + threads*ipt - 1) // (threads*ipt)
copy_blocked[blocks_b, threads](src, dst, ipt)
cp.cuda.Device().synchronize()
cp.testing.assert_array_equal(src, dst); print('blocked OK')

<a id="3"></a>
## 3. Coalesced Copy (최적화) — 연습

**grid-stride 루프**로 바꿉니다: warp의 인접 스레드가 **인접 주소**를 읽도록(연속 접근), 그다음 `gridsize(1)`만큼
건너뛰며 반복. 이러면 메모리 트랜잭션이 합쳐져(coalesced) 대역폭을 제대로 씁니다.

In [ ]:
@cuda.jit
def copy_coalesced(src, dst):
    # TODO: i = cuda.grid(1); stride = cuda.gridsize(1)
    #       while i < src.size: dst[i] = src[i]; i += stride
    pass

blocks_c = 1024   # grid-stride라 블록 수는 자유(충분히 크게)
# copy_coalesced[blocks_c, threads](src, dst)
# cp.cuda.Device().synchronize(); cp.testing.assert_array_equal(src, dst); print('coalesced OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def copy_coalesced(src, dst):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    while i < src.size:
        dst[i] = src[i]
        i += stride

copy_coalesced[blocks_c, threads](src, dst)
cp.cuda.Device().synchronize()
cp.testing.assert_array_equal(src, dst); print('coalesced OK')
```
</details>

<a id="4"></a>
## 4. 성능 비교

두 커널을 `bench`로 비교합니다. 같은 복사인데 **접근 패턴**만으로 차이가 납니다(연속 접근이 보통 더 빠름).

In [ ]:
def run_blocked():   copy_blocked[blocks_b, threads](src, dst, ipt)
def run_coalesced(): copy_coalesced[blocks_c, threads](src, dst)
print_bench(bench(run_blocked,   n_repeat=20, n_warmup=5, name='blocked'))
print_bench(bench(run_coalesced, n_repeat=20, n_warmup=5, name='coalesced'))
print('speedup:', round(gpu_ms(bench(run_blocked))/gpu_ms(bench(run_coalesced)), 2))

<a id="5"></a>
## 5. 파라미터 스윕 (occupancy)

`threads_per_block`을 바꾸며 측정합니다. occupancy·메모리 효율이 달라져 최적점이 생깁니다(07의 7절).

In [ ]:
for threads in [64, 128, 256, 512, 1024]:
    bl = 2048
    def run(t=threads, b=bl): copy_coalesced[b, t](src, dst)
    print_bench(bench(run, n_repeat=20, n_warmup=5, name=f'threads={threads}'))

<a id="6"></a>
## 6. (선택) Nsight Compute 프로파일

<details><summary>펼쳐 보기 — ncu로 메모리 처리량 직접 보기 (GTC 06 워크플로)</summary>

프로파일링은 별도 프로세스 실행이 필요해, 스크립트로 저장 후 커맨드라인 `ncu`로 측정합니다.
강의장에 **Nsight Compute(`ncu`)** 가 있을 때만 동작합니다.

```python
# 주피터 셀 맨 위:  %%writefile copy_bench.py
from numba import cuda; import cupy as cp, numpy as np
@cuda.jit
def copy_coalesced(src,dst):
    i=cuda.grid(1); s=cuda.gridsize(1)
    while i<src.size: dst[i]=src[i]; i+=s
N=1<<26; src=cp.arange(N,dtype=cp.float32); dst=cp.empty_like(src)
copy_coalesced[2048,256](src,dst); cp.cuda.Device().synchronize()
```
```bash
ncu -f --kernel-name regex:copy_coalesced --set full -o copy python copy_bench.py
ncu --import copy.ncu-rep --csv   # 또는 nsightful로 노트북에 표시
```
리포트의 **Memory Workload** 에서 blocked 대비 coalesced의 처리량(throughput)이 높은 것을 확인하세요.
</details>

## 🧪 추가 연습 & 실험

**실험 — items_per_thread 스윕**: blocked copy에서 스레드당 처리량을 바꿔 측정해 보세요(접근 패턴·occupancy 영향).

In [ ]:
for ipt in [1, 4, 8, 16, 32]:
    bb = (N + threads*ipt - 1)//(threads*ipt)
    def run(b=bb, k=ipt): copy_blocked[b, threads](src, dst, k)
    print('ipt', ipt, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

**연습 — saxpy Numba 커널**: `y = a*x + b` 를 grid-stride로 작성하세요(coalesced).

In [ ]:
@cuda.jit
def saxpy(x, y, a, b):
    # TODO: i=cuda.grid(1); s=cuda.gridsize(1); while i<x.size: y[i]=a*x[i]+b; i+=s
    pass
xx = cp.random.rand(1<<22, dtype=cp.float32); yy = cp.empty_like(xx)
# saxpy[1024,256](xx,yy,np.float32(2),np.float32(1)); cp.cuda.Device().synchronize()
# cp.testing.assert_allclose(cp.asnumpy(yy), 2*cp.asnumpy(xx)+1, rtol=1e-5); print('OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def saxpy(x, y, a, b):
    i = cuda.grid(1); s = cuda.gridsize(1)
    while i < x.size:
        y[i] = a*x[i] + b; i += s
saxpy[1024,256](xx, yy, np.float32(2), np.float32(1)); cp.cuda.Device().synchronize()
cp.testing.assert_allclose(cp.asnumpy(yy), 2*cp.asnumpy(xx)+1, rtol=1e-5); print('OK')
```
</details>

**연습 — 2D naive transpose**: 행을 읽어 열에 쓰는 전치 커널. 쓰기가 **비연속(non-coalesced)** 임을 관찰하세요(11에서 타일로 개선).

In [ ]:
@cuda.jit
def transpose_naive(a, out, n):
    # TODO: i,j = cuda.grid(2); if i<n and j<n: out[j,i] = a[i,j]
    pass
n=2048; A=cp.random.rand(n,n,dtype=cp.float32); T=cp.empty_like(A)
tpb=(16,16); bpg=(math.ceil(n/16), math.ceil(n/16))
# transpose_naive[bpg,tpb](A,T,np.int32(n)); cp.cuda.Device().synchronize()
# cp.testing.assert_array_equal(cp.asnumpy(T), cp.asnumpy(A).T); print('OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def transpose_naive(a, out, n):
    i, j = cuda.grid(2)
    if i < n and j < n:
        out[j, i] = a[i, j]    # 읽기는 연속, 쓰기는 비연속
transpose_naive[bpg,tpb](A,T,np.int32(n)); cp.cuda.Device().synchronize()
cp.testing.assert_array_equal(cp.asnumpy(T), cp.asnumpy(A).T); print('OK')
```
</details>

<a id="7"></a>
## 7. 체크포인트

- [ ] `@cuda.jit`+`cuda.grid`로 커널을 작성·런치했다
- [ ] blocked vs coalesced **메모리 접근 패턴**의 성능 차이를 측정했다
- [ ] grid-stride 루프로 coalesced 복사를 구현했다
- [ ] threads_per_block 스윕으로 occupancy 영향을 봤다
- [ ] (선택) ncu로 메모리 처리량을 확인했다

다음: **`09_numba_histogram`** — atomic·공유메모리·동기화로 히스토그램을 단계적으로 최적화합니다.